# SAM + FPN + FCOS Detection Head Training (Colab)

Self-contained notebook for training the SAM-based document layout detector on DocLayNet.

The SAM ViT-B encoder is frozen (loaded from DeepSeek OCR 2 weights).
Only the FPN and FCOS head parameters (~6.4M) are trained.

**Runtime setup:** Go to Runtime > Change runtime type > select **T4 GPU** (free) or **A100** (Pro).

In [ ]:
# ---- Install dependencies ----
!pip install -q torch torchvision safetensors huggingface_hub pillow

In [ ]:
# ---- Mount Google Drive for checkpoint persistence ----
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/VisionPDF_checkpoints"

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {SAVE_DIR}")

In [ ]:
# ---- Check GPU ----
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {mem_gb:.1f} GB")

In [ ]:
# =========================================================================
# Configuration
# =========================================================================

DOCLAYNET_LABELS = [
    "Caption",
    "Footnote",
    "Formula",
    "List-item",
    "Page-footer",
    "Page-header",
    "Picture",
    "Section-header",
    "Table",
    "Text",
    "Title",
]

NUM_CLASSES = len(DOCLAYNET_LABELS)  # 11

DEEPSEEK_OCR2_MODEL = "deepseek-ai/DeepSeek-OCR-2"

DOCLAYNET_DIR = "/content/DocLayNet"
DOCLAYNET_ZIP_URL = (
    "https://codait-cos-dax.s3.us.cloud-object-storage.appdomain.cloud"
    "/dax-doclaynet/1.0.0/DocLayNet_core.zip"
)

In [ ]:
# =========================================================================
# Hyperparameters -- adjust these as needed
# =========================================================================

EPOCHS = 12
BATCH_SIZE = 16
LEARNING_RATE = 4e-4
WEIGHT_DECAY = 0.05
NUM_WORKERS = 2
TARGET_SIZE = 1024

In [ ]:
# =========================================================================
# SAM ViT-B Encoder
# =========================================================================

import math
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch.utils.data import Dataset, DataLoader


class _LayerNorm2d(nn.Module):
    def __init__(self, num_channels: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return self.weight[:, None, None] * x + self.bias[:, None, None]


class _MLPBlock(nn.Module):
    def __init__(self, embedding_dim, mlp_dim):
        super().__init__()
        self.lin1 = nn.Linear(embedding_dim, mlp_dim)
        self.lin2 = nn.Linear(mlp_dim, embedding_dim)
        self.act = nn.GELU()

    def forward(self, x):
        return self.lin2(self.act(self.lin1(x)))


def _get_abs_pos(abs_pos, tgt_size):
    src_size = abs_pos.size(1)
    if src_size != tgt_size:
        old = abs_pos.permute(0, 3, 1, 2).to(torch.float32)
        new = F.interpolate(old, size=(tgt_size, tgt_size), mode="bicubic",
                            antialias=True, align_corners=False).to(abs_pos.dtype)
        return new.permute(0, 2, 3, 1)
    return abs_pos


def _get_rel_pos(q_size, k_size, rel_pos):
    max_rel_dist = int(2 * max(q_size, k_size) - 1)
    if rel_pos.shape[0] != max_rel_dist:
        rp = rel_pos.to(torch.float32)
        rp = F.interpolate(
            rp.reshape(1, rp.shape[0], -1).permute(0, 2, 1),
            size=max_rel_dist, mode="linear",
        ).to(rel_pos.dtype)
        rel_pos = rp.reshape(-1, max_rel_dist).permute(1, 0)
    q_coords = torch.arange(q_size, device=rel_pos.device)[:, None] * max(k_size / q_size, 1.0)
    k_coords = torch.arange(k_size, device=rel_pos.device)[None, :] * max(q_size / k_size, 1.0)
    relative_coords = (q_coords - k_coords) + (k_size - 1) * max(q_size / k_size, 1.0)
    return rel_pos[relative_coords.long()]


class _Attention(nn.Module):
    def __init__(self, dim, num_heads=12, qkv_bias=True,
                 use_rel_pos=False, rel_pos_zero_init=True, input_size=None):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)
        self.use_rel_pos = use_rel_pos
        if use_rel_pos:
            head_dim = dim // num_heads
            self.rel_pos_h = nn.Parameter(torch.zeros(2 * input_size[0] - 1, head_dim))
            self.rel_pos_w = nn.Parameter(torch.zeros(2 * input_size[1] - 1, head_dim))

    def forward(self, x):
        B, H, W, _ = x.shape
        qkv = self.qkv(x).reshape(B, H * W, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.reshape(3, B * self.num_heads, H * W, -1).unbind(0)
        attn_bias = None
        if self.use_rel_pos:
            Rh = _get_rel_pos(H, H, self.rel_pos_h)
            Rw = _get_rel_pos(W, W, self.rel_pos_w)
            r_q = q.view(B * self.num_heads, H, W, -1)
            rel_h = torch.einsum("bhwc,hkc->bhwk", r_q, Rh).unsqueeze(-1)
            rel_w = torch.einsum("bhwc,wkc->bhwk", r_q, Rw).unsqueeze(-2)
            attn_bias = (rel_h + rel_w).reshape(B * self.num_heads, H * W, H * W)
            q = q.view(B, self.num_heads, H * W, -1)
            k = k.view(B, self.num_heads, H * W, -1)
            v = v.view(B, self.num_heads, H * W, -1)
            attn_bias = attn_bias.view(B, self.num_heads, H * W, H * W)
            x = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias)
        else:
            q = q.view(B, self.num_heads, H * W, -1)
            k = k.view(B, self.num_heads, H * W, -1)
            v = v.view(B, self.num_heads, H * W, -1)
            x = F.scaled_dot_product_attention(q, k, v)
        x = x.view(B, self.num_heads, H, W, -1).permute(0, 2, 3, 1, 4).reshape(B, H, W, -1)
        return self.proj(x)


def _window_partition(x, window_size):
    B, H, W, C = x.shape
    pad_h = (window_size - H % window_size) % window_size
    pad_w = (window_size - W % window_size) % window_size
    if pad_h > 0 or pad_w > 0:
        x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
    Hp, Wp = H + pad_h, W + pad_w
    x = x.view(B, Hp // window_size, window_size, Wp // window_size, window_size, C)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C), (Hp, Wp)


def _window_unpartition(windows, window_size, pad_hw, hw):
    Hp, Wp = pad_hw
    H, W = hw
    B = windows.shape[0] // (Hp * Wp // window_size // window_size)
    x = windows.view(B, Hp // window_size, Wp // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, Hp, Wp, -1)
    if Hp > H or Wp > W:
        x = x[:, :H, :W, :].contiguous()
    return x


class _Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=True,
                 use_rel_pos=False, rel_pos_zero_init=True,
                 window_size=0, input_size=None):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = _Attention(
            dim, num_heads=num_heads, qkv_bias=qkv_bias,
            use_rel_pos=use_rel_pos, rel_pos_zero_init=rel_pos_zero_init,
            input_size=input_size if window_size == 0 else (window_size, window_size),
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = _MLPBlock(dim, int(dim * mlp_ratio))
        self.window_size = window_size

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        if self.window_size > 0:
            H, W = x.shape[1], x.shape[2]
            x, pad_hw = _window_partition(x, self.window_size)
        x = self.attn(x)
        if self.window_size > 0:
            x = _window_unpartition(x, self.window_size, pad_hw, (H, W))
        x = shortcut + x
        return x + self.mlp(self.norm2(x))


class _PatchEmbed(nn.Module):
    def __init__(self, kernel_size=(16, 16), stride=(16, 16),
                 in_chans=3, embed_dim=768):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim,
                              kernel_size=kernel_size, stride=stride)

    def forward(self, x):
        return self.proj(x).permute(0, 2, 3, 1)


class SAMEncoder(nn.Module):
    """Minimal SAM ViT-B encoder matching DeepSeek OCR 2 architecture."""

    def __init__(self):
        super().__init__()
        embed_dim = 768
        depth = 12
        num_heads = 12
        img_size = 1024
        patch_size = 16
        out_chans = 256
        global_attn_indexes = (2, 5, 8, 11)

        self.patch_embed = _PatchEmbed(
            kernel_size=(patch_size, patch_size),
            stride=(patch_size, patch_size),
            in_chans=3, embed_dim=embed_dim,
        )
        feat_size = img_size // patch_size
        self.pos_embed = nn.Parameter(
            torch.zeros(1, feat_size, feat_size, embed_dim)
        )
        self.blocks = nn.ModuleList()
        for i in range(depth):
            self.blocks.append(_Block(
                dim=embed_dim, num_heads=num_heads, mlp_ratio=4.0,
                qkv_bias=True, use_rel_pos=True, rel_pos_zero_init=True,
                window_size=14 if i not in global_attn_indexes else 0,
                input_size=(feat_size, feat_size),
            ))
        self.neck = nn.Sequential(
            nn.Conv2d(embed_dim, out_chans, kernel_size=1, bias=False),
            _LayerNorm2d(out_chans),
            nn.Conv2d(out_chans, out_chans, kernel_size=3, padding=1, bias=False),
            _LayerNorm2d(out_chans),
        )
        self.net_2 = nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1, bias=False)
        self.net_3 = nn.Conv2d(512, 896, kernel_size=3, stride=2, padding=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x)
        x = x + _get_abs_pos(self.pos_embed, x.size(1))
        for blk in self.blocks:
            x = blk(x)
        x = self.neck(x.permute(0, 3, 1, 2))
        x = self.net_2(x)
        x = self.net_3(x)
        return x


def load_sam_encoder(model_path: str, device: torch.device) -> SAMEncoder:
    """Load SAM encoder weights from the DeepSeek OCR 2 checkpoint."""
    from huggingface_hub import snapshot_download

    if Path(model_path).is_dir():
        model_dir = Path(model_path)
    else:
        print(f"Downloading model files from {model_path} (vision weights only)...")
        model_dir = Path(snapshot_download(
            model_path,
            allow_patterns=["*.safetensors", "*.json", "*.bin"],
        ))

    state_dict: Dict[str, torch.Tensor] = {}
    safetensor_files = list(model_dir.glob("*.safetensors"))

    if safetensor_files:
        from safetensors.torch import load_file
        for sf in safetensor_files:
            sd = load_file(str(sf), device="cpu")
            for k, v in sd.items():
                if "sam_model" in k:
                    state_dict[k] = v
    else:
        bin_files = list(model_dir.glob("*.bin"))
        for bf in bin_files:
            sd = torch.load(str(bf), map_location="cpu")
            for k, v in sd.items():
                if "sam_model" in k:
                    state_dict[k] = v

    cleaned: Dict[str, torch.Tensor] = {}
    for k, v in state_dict.items():
        new_key = k
        for prefix in ("model.sam_model.", "sam_model."):
            if new_key.startswith(prefix):
                new_key = new_key[len(prefix):]
                break
        cleaned[new_key] = v

    encoder = SAMEncoder()
    missing, unexpected = encoder.load_state_dict(cleaned, strict=False)
    if missing:
        print(f"Warning: missing keys in SAM encoder: {missing[:5]}{'...' if len(missing) > 5 else ''}")
    if unexpected:
        print(f"Warning: unexpected keys: {unexpected[:5]}{'...' if len(unexpected) > 5 else ''}")

    encoder = encoder.to(device=device, dtype=torch.float16).eval()
    return encoder


print("Encoder defined.")

In [ ]:
# =========================================================================
# Detector: MultiScaleSAMEncoder + FPN + FCOS Head
# =========================================================================

class MultiScaleSAMEncoder(nn.Module):
    """Wraps SAMEncoder to return multi-scale feature maps (p3, p4, p5)."""

    def __init__(self, sam_encoder: SAMEncoder):
        super().__init__()
        self.sam = sam_encoder

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        s = self.sam
        x = s.patch_embed(x)
        x = x + _get_abs_pos(s.pos_embed, x.size(1))
        for blk in s.blocks:
            x = blk(x)
        p3 = s.neck(x.permute(0, 3, 1, 2))
        p4 = s.net_2(p3)
        p5 = s.net_3(p4)
        return {"p3": p3, "p4": p4, "p5": p5}


class FPN(nn.Module):
    """Top-down FPN fusing P3/P4/P5 to 256-channel maps."""

    def __init__(self, in_channels_p3=256, in_channels_p4=512,
                 in_channels_p5=896, out_channels=256):
        super().__init__()
        self.lateral_p5 = nn.Conv2d(in_channels_p5, out_channels, 1)
        self.lateral_p4 = nn.Conv2d(in_channels_p4, out_channels, 1)
        self.lateral_p3 = nn.Conv2d(in_channels_p3, out_channels, 1)
        self.smooth_p4 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.smooth_p3 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, features):
        p3_in, p4_in, p5_in = features["p3"], features["p4"], features["p5"]
        p5 = self.lateral_p5(p5_in)
        p4 = self.lateral_p4(p4_in) + F.interpolate(
            p5, size=p4_in.shape[2:], mode="bilinear", align_corners=False)
        p4 = self.smooth_p4(p4)
        p3 = self.lateral_p3(p3_in) + F.interpolate(
            p4, size=p3_in.shape[2:], mode="bilinear", align_corners=False)
        p3 = self.smooth_p3(p3)
        return {"p3": p3, "p4": p4, "p5": p5}


class FCOSHead(nn.Module):
    """Anchor-free FCOS detection head."""

    def __init__(self, in_channels=256, num_convs=4):
        super().__init__()
        cls_tower, reg_tower = [], []
        for _ in range(num_convs):
            cls_tower.extend([
                nn.Conv2d(in_channels, in_channels, 3, padding=1, bias=False),
                nn.GroupNorm(32, in_channels),
                nn.ReLU(inplace=True)])
            reg_tower.extend([
                nn.Conv2d(in_channels, in_channels, 3, padding=1, bias=False),
                nn.GroupNorm(32, in_channels),
                nn.ReLU(inplace=True)])
        self.cls_tower = nn.Sequential(*cls_tower)
        self.reg_tower = nn.Sequential(*reg_tower)
        self.cls_logits = nn.Conv2d(in_channels, NUM_CLASSES, 3, padding=1)
        self.bbox_pred = nn.Conv2d(in_channels, 4, 3, padding=1)
        self.centerness = nn.Conv2d(in_channels, 1, 3, padding=1)
        self._init_weights()

    def _init_weights(self):
        for modules in [self.cls_tower, self.reg_tower]:
            for m in modules.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.normal_(m.weight, std=0.01)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
        nn.init.normal_(self.cls_logits.weight, std=0.01)
        nn.init.constant_(self.cls_logits.bias, -math.log((1 - 0.01) / 0.01))
        nn.init.normal_(self.bbox_pred.weight, std=0.01)
        nn.init.zeros_(self.bbox_pred.bias)
        nn.init.normal_(self.centerness.weight, std=0.01)
        nn.init.zeros_(self.centerness.bias)

    def forward(self, x):
        cls_feat = self.cls_tower(x)
        reg_feat = self.reg_tower(x)
        return (
            self.cls_logits(cls_feat),
            torch.exp(self.bbox_pred(reg_feat)),
            self.centerness(cls_feat),
        )


class SAMDetector(nn.Module):
    """Frozen SAM encoder + trainable FPN + trainable FCOS head."""

    LEVEL_RANGES = {"p3": (0, 64), "p4": (64, 256), "p5": (256, 1e8)}
    STRIDES = {"p3": 16, "p4": 32, "p5": 64}

    def __init__(self, sam_encoder: SAMEncoder):
        super().__init__()
        self.backbone = MultiScaleSAMEncoder(sam_encoder)
        self.fpn = FPN()
        self.head = FCOSHead()

    def forward(self, images):
        features = self.backbone(images)
        fpn_features = self.fpn(features)
        outputs = {}
        for level_name, feat in fpn_features.items():
            outputs[level_name] = self.head(feat)
        return outputs


print("Detector defined.")

In [ ]:
# =========================================================================
# FCOS target assignment and losses
# =========================================================================

def compute_fcos_targets(gt_boxes, gt_labels, fpn_levels, strides, level_ranges):
    """Assign FCOS regression/classification targets for one image."""
    device = gt_boxes.device
    targets = {}

    for level_name, (_, H, W) in fpn_levels.items():
        stride = strides[level_name]
        lo, hi = level_ranges[level_name]

        cls_target = torch.zeros(H, W, dtype=torch.long, device=device)
        reg_target = torch.zeros(H, W, 4, dtype=torch.float32, device=device)
        cent_target = torch.zeros(H, W, dtype=torch.float32, device=device)

        if gt_boxes.numel() == 0:
            targets[level_name] = (cls_target, reg_target, cent_target)
            continue

        shifts_y = (torch.arange(H, device=device).float() + 0.5) * stride
        shifts_x = (torch.arange(W, device=device).float() + 0.5) * stride
        sy, sx = torch.meshgrid(shifts_y, shifts_x, indexing="ij")

        N = gt_boxes.shape[0]
        sx_exp = sx.unsqueeze(-1).expand(H, W, N)
        sy_exp = sy.unsqueeze(-1).expand(H, W, N)

        l = sx_exp - gt_boxes[:, 0].view(1, 1, N)
        t = sy_exp - gt_boxes[:, 1].view(1, 1, N)
        r = gt_boxes[:, 2].view(1, 1, N) - sx_exp
        b = gt_boxes[:, 3].view(1, 1, N) - sy_exp
        ltrb = torch.stack([l, t, r, b], dim=-1)

        inside = ltrb.min(dim=-1).values > 0
        max_ltrb = ltrb.max(dim=-1).values
        in_range = (max_ltrb >= lo) & (max_ltrb < hi)
        valid = inside & in_range

        areas = (gt_boxes[:, 2] - gt_boxes[:, 0]) * (gt_boxes[:, 3] - gt_boxes[:, 1])
        areas_exp = areas.view(1, 1, N).expand(H, W, N)
        areas_exp = torch.where(valid, areas_exp, torch.tensor(1e8, device=device))
        min_area_idx = areas_exp.argmin(dim=-1)

        has_gt = valid.any(dim=-1)

        for hy in range(H):
            for wx in range(W):
                if not has_gt[hy, wx]:
                    continue
                gi = min_area_idx[hy, wx]
                cls_target[hy, wx] = gt_labels[gi] + 1
                l_val = ltrb[hy, wx, gi, 0]
                t_val = ltrb[hy, wx, gi, 1]
                r_val = ltrb[hy, wx, gi, 2]
                b_val = ltrb[hy, wx, gi, 3]
                reg_target[hy, wx] = torch.tensor([l_val, t_val, r_val, b_val])
                lr_min = min(l_val, r_val)
                lr_max = max(l_val, r_val)
                tb_min = min(t_val, b_val)
                tb_max = max(t_val, b_val)
                if lr_max > 0 and tb_max > 0:
                    cent_target[hy, wx] = math.sqrt(
                        (lr_min / lr_max) * (tb_min / tb_max)
                    )

        targets[level_name] = (cls_target, reg_target, cent_target)

    return targets


def focal_loss(pred, target, alpha=0.25, gamma=2.0):
    """Multi-class focal loss. pred: (N, C) logits, target: (N,) long."""
    num_classes = pred.shape[1]
    t = torch.zeros_like(pred)
    fg_mask = target > 0
    if fg_mask.any():
        t[fg_mask, target[fg_mask] - 1] = 1.0
    p = pred.sigmoid()
    ce = F.binary_cross_entropy_with_logits(pred, t, reduction="none")
    pt = torch.where(t == 1, p, 1 - p)
    loss = alpha * (1 - pt) ** gamma * ce
    return loss.sum() / max(fg_mask.sum().item(), 1)


def giou_loss(pred_ltrb, target_ltrb):
    """GIoU loss on LTRB distances."""
    if pred_ltrb.numel() == 0:
        return pred_ltrb.sum() * 0.0
    pred_area = (pred_ltrb[:, 0] + pred_ltrb[:, 2]) * (pred_ltrb[:, 1] + pred_ltrb[:, 3])
    target_area = (target_ltrb[:, 0] + target_ltrb[:, 2]) * (target_ltrb[:, 1] + target_ltrb[:, 3])
    w_inter = torch.min(pred_ltrb[:, 0], target_ltrb[:, 0]) + torch.min(pred_ltrb[:, 2], target_ltrb[:, 2])
    h_inter = torch.min(pred_ltrb[:, 1], target_ltrb[:, 1]) + torch.min(pred_ltrb[:, 3], target_ltrb[:, 3])
    inter = w_inter.clamp(min=0) * h_inter.clamp(min=0)
    union = pred_area + target_area - inter + 1e-7
    iou = inter / union
    w_enclose = torch.max(pred_ltrb[:, 0], target_ltrb[:, 0]) + torch.max(pred_ltrb[:, 2], target_ltrb[:, 2])
    h_enclose = torch.max(pred_ltrb[:, 1], target_ltrb[:, 1]) + torch.max(pred_ltrb[:, 3], target_ltrb[:, 3])
    enclose_area = w_enclose * h_enclose + 1e-7
    giou = iou - (enclose_area - union) / enclose_area
    return (1 - giou).mean()


print("Losses defined.")

In [ ]:
# =========================================================================
# DocLayNet Dataset
# =========================================================================

import json
import random
import shutil
import tempfile
import urllib.request
import zipfile

_LABEL_TO_IDX = {name: i for i, name in enumerate(DOCLAYNET_LABELS)}
_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def download_doclaynet(target_dir):
    """Download and extract DocLayNet if not present."""
    target_dir = Path(target_dir)
    if (target_dir / "COCO" / "train.json").exists():
        print(f"DocLayNet already present at {target_dir}")
        return target_dir

    target_dir.mkdir(parents=True, exist_ok=True)
    print("Downloading DocLayNet core dataset (~28 GB)...")
    print(f"  Source: {DOCLAYNET_ZIP_URL}")

    zip_path = Path(tempfile.mktemp(suffix=".zip"))
    try:
        def _report(block_num, block_size, total_size):
            downloaded = block_num * block_size
            if total_size > 0:
                pct = min(downloaded / total_size * 100, 100)
                mb = downloaded / 1e6
                total_mb = total_size / 1e6
                print(f"\r  {mb:.0f}/{total_mb:.0f} MB ({pct:.1f}%)", end="", flush=True)

        urllib.request.urlretrieve(DOCLAYNET_ZIP_URL, zip_path, reporthook=_report)
        print()
        print("Extracting...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(target_dir)

        for subdir_name in ["DocLayNet_core", "DocLayNet"]:
            nested = target_dir / subdir_name
            if nested.is_dir() and (nested / "COCO").exists():
                for item in nested.iterdir():
                    dest = target_dir / item.name
                    if not dest.exists():
                        shutil.move(str(item), str(dest))
                shutil.rmtree(nested, ignore_errors=True)
                break
    finally:
        if zip_path.exists():
            zip_path.unlink(missing_ok=True)

    print(f"DocLayNet ready at {target_dir}")
    return target_dir


def _load_coco_annotations(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)
    cat_id_to_name = {cat["id"]: cat["name"] for cat in coco.get("categories", [])}
    img_id_to_anns = {}
    for ann in coco.get("annotations", []):
        img_id_to_anns.setdefault(ann["image_id"], []).append(ann)
    return coco["images"], cat_id_to_name, img_id_to_anns


class DocLayNetDataset(Dataset):
    """PyTorch Dataset for DocLayNet detection training."""

    def __init__(self, split="train", target_size=1024, augment=True, data_dir=None):
        self.target_size = target_size
        self.augment = augment and (split == "train")
        data_dir = Path(data_dir or DOCLAYNET_DIR)
        json_path = data_dir / "COCO" / f"{split}.json"

        if not json_path.exists():
            download_doclaynet(data_dir)
        if not json_path.exists():
            raise FileNotFoundError(f"DocLayNet annotation not found: {json_path}")

        self.images_dir = data_dir / "PNG"
        self.image_list, self.cat_id_to_name, self.img_id_to_anns = (
            _load_coco_annotations(json_path)
        )
        self.cat_id_to_idx = {}
        for coco_id, name in self.cat_id_to_name.items():
            if name in _LABEL_TO_IDX:
                self.cat_id_to_idx[coco_id] = _LABEL_TO_IDX[name]

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_info = self.image_list[idx]
        img_id = img_info["id"]
        file_name = img_info["file_name"]
        image = Image.open(self.images_dir / file_name).convert("RGB")

        anns = self.img_id_to_anns.get(img_id, [])
        boxes, labels = [], []
        for ann in anns:
            coco_cat_id = ann["category_id"]
            if coco_cat_id not in self.cat_id_to_idx:
                continue
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(self.cat_id_to_idx[coco_cat_id])

        boxes = np.array(boxes, dtype=np.float32).reshape(-1, 4)
        labels = np.array(labels, dtype=np.int64)

        if self.augment:
            image, boxes = self._augment(image, boxes)
        image, boxes, scale = self._resize_and_pad(image, boxes)

        arr = np.array(image, dtype=np.float32) / 255.0
        arr = (arr - _MEAN) / _STD
        tensor = torch.from_numpy(arr).permute(2, 0, 1)
        return {
            "image": tensor,
            "boxes": torch.from_numpy(boxes).float(),
            "labels": torch.from_numpy(labels).long(),
            "scale": scale,
        }

    def _augment(self, image, boxes):
        w, h = image.size
        if random.random() < 0.5:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            if boxes.shape[0] > 0:
                x1 = w - boxes[:, 2]
                x2 = w - boxes[:, 0]
                boxes[:, 0] = x1
                boxes[:, 2] = x2
        arr = np.array(image, dtype=np.float32)
        brightness = random.uniform(0.8, 1.2)
        contrast = random.uniform(0.8, 1.2)
        arr = arr * contrast + (brightness - 1.0) * 128
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        return Image.fromarray(arr), boxes

    def _resize_and_pad(self, image, boxes):
        w, h = image.size
        scale = self.target_size / max(w, h)
        new_w, new_h = int(w * scale), int(h * scale)
        image = image.resize((new_w, new_h), Image.BILINEAR)
        canvas = Image.new("RGB", (self.target_size, self.target_size), (0, 0, 0))
        canvas.paste(image, (0, 0))
        if boxes.shape[0] > 0:
            boxes = boxes * scale
            boxes[:, 0] = np.clip(boxes[:, 0], 0, new_w)
            boxes[:, 1] = np.clip(boxes[:, 1], 0, new_h)
            boxes[:, 2] = np.clip(boxes[:, 2], 0, new_w)
            boxes[:, 3] = np.clip(boxes[:, 3], 0, new_h)
        return canvas, boxes, scale


def collate_fn(batch):
    images = torch.stack([item["image"] for item in batch])
    boxes = [item["boxes"] for item in batch]
    labels = [item["labels"] for item in batch]
    scales = torch.tensor([item["scale"] for item in batch])
    return {"images": images, "boxes": boxes, "labels": labels, "scales": scales}


print("Dataset defined.")

In [ ]:
# =========================================================================
# Download DocLayNet (this takes a while on the first run)
# =========================================================================

download_doclaynet(DOCLAYNET_DIR)

In [ ]:
# =========================================================================
# Build model and data loaders
# =========================================================================

import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

print(f"Loading SAM encoder from {DEEPSEEK_OCR2_MODEL}...")
sam_encoder = load_sam_encoder(DEEPSEEK_OCR2_MODEL, device)

detector = SAMDetector(sam_encoder).to(device)

for param in detector.backbone.parameters():
    param.requires_grad = False

trainable_params = list(detector.fpn.parameters()) + list(detector.head.parameters())
num_params = sum(p.numel() for p in trainable_params if p.requires_grad)
print(f"Trainable parameters: {num_params:,} ({num_params/1e6:.1f}M)")

print("Loading DocLayNet train/val splits...")
train_ds = DocLayNetDataset("train", target_size=TARGET_SIZE, augment=True, data_dir=DOCLAYNET_DIR)
val_ds = DocLayNetDataset("val", target_size=TARGET_SIZE, augment=False, data_dir=DOCLAYNET_DIR)
print(f"  Train: {len(train_ds)} images")
print(f"  Val:   {len(val_ds)} images")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True,
)

optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("Ready to train.")

In [ ]:
# =========================================================================
# Training loop
# =========================================================================

save_path = os.path.join(SAVE_DIR, "sam_doclaynet_head.pt")
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ---- Train ----
    detector.fpn.train()
    detector.head.train()
    total_loss_sum = 0.0
    num_batches = 0

    for batch_idx, batch in enumerate(train_loader):
        images = batch["images"].to(device, dtype=torch.float16)
        gt_boxes_list = batch["boxes"]
        gt_labels_list = batch["labels"]

        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = detector(images)

        batch_cls_loss = 0.0
        batch_reg_loss = 0.0
        batch_cent_loss = 0.0
        B = images.shape[0]

        for b in range(B):
            gt_boxes = gt_boxes_list[b].to(device)
            gt_labels = gt_labels_list[b].to(device)

            fpn_levels = {}
            for level_name, (cls_logits, bbox_pred, centerness) in outputs.items():
                _, C, H, W = cls_logits.shape
                fpn_levels[level_name] = (C, H, W)

            targets = compute_fcos_targets(
                gt_boxes, gt_labels, fpn_levels,
                SAMDetector.STRIDES, SAMDetector.LEVEL_RANGES,
            )

            for level_name in outputs:
                cls_logits, bbox_pred, centerness = outputs[level_name]
                cls_target, reg_target, cent_target = targets[level_name]

                cls_pred_flat = cls_logits[b].permute(1, 2, 0).reshape(-1, NUM_CLASSES)
                cls_target_flat = cls_target.reshape(-1)
                batch_cls_loss = batch_cls_loss + focal_loss(
                    cls_pred_flat.float(), cls_target_flat
                )

                fg_mask = cls_target.reshape(-1) > 0
                if fg_mask.any():
                    pred_ltrb = bbox_pred[b].permute(1, 2, 0).reshape(-1, 4)[fg_mask]
                    tgt_ltrb = reg_target.reshape(-1, 4)[fg_mask]
                    batch_reg_loss = batch_reg_loss + giou_loss(
                        pred_ltrb.float(), tgt_ltrb
                    )

                    pred_cent = centerness[b, 0].reshape(-1)[fg_mask]
                    tgt_cent = cent_target.reshape(-1)[fg_mask]
                    batch_cent_loss = batch_cent_loss + F.binary_cross_entropy_with_logits(
                        pred_cent.float(), tgt_cent, reduction="mean"
                    )

        loss = (batch_cls_loss + batch_reg_loss + batch_cent_loss) / B

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(detector.fpn.parameters()) + list(detector.head.parameters()),
            max_norm=1.0,
        )
        optimizer.step()

        total_loss_sum += loss.item()
        num_batches += 1

        if batch_idx % 50 == 0:
            cls_val = batch_cls_loss.item() / B if isinstance(batch_cls_loss, torch.Tensor) else 0
            reg_val = batch_reg_loss.item() / B if isinstance(batch_reg_loss, torch.Tensor) else 0
            cent_val = batch_cent_loss.item() / B if isinstance(batch_cent_loss, torch.Tensor) else 0
            print(
                f"  Epoch {epoch} [{batch_idx}/{len(train_loader)}] "
                f"loss={loss.item():.4f} "
                f"(cls={cls_val:.4f} reg={reg_val:.4f} cent={cent_val:.4f})"
            )

    train_loss = total_loss_sum / max(num_batches, 1)
    scheduler.step()

    # ---- Validate ----
    detector.eval()
    val_loss_sum = 0.0
    val_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            images = batch["images"].to(device, dtype=torch.float16)
            gt_boxes_list = batch["boxes"]
            gt_labels_list = batch["labels"]

            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = detector(images)

            B = images.shape[0]
            batch_cls_loss = 0.0
            batch_reg_loss = 0.0
            batch_cent_loss = 0.0

            for b in range(B):
                gt_boxes = gt_boxes_list[b].to(device)
                gt_labels = gt_labels_list[b].to(device)

                fpn_levels = {}
                for level_name, (cls_logits, bbox_pred, centerness) in outputs.items():
                    _, C, H, W = cls_logits.shape
                    fpn_levels[level_name] = (C, H, W)

                targets = compute_fcos_targets(
                    gt_boxes, gt_labels, fpn_levels,
                    SAMDetector.STRIDES, SAMDetector.LEVEL_RANGES,
                )

                for level_name in outputs:
                    cls_logits, bbox_pred, centerness = outputs[level_name]
                    cls_target, reg_target, cent_target = targets[level_name]

                    cls_pred_flat = cls_logits[b].permute(1, 2, 0).reshape(-1, NUM_CLASSES)
                    cls_target_flat = cls_target.reshape(-1)
                    batch_cls_loss += focal_loss(
                        cls_pred_flat.float(), cls_target_flat
                    ).item()

                    fg_mask = cls_target.reshape(-1) > 0
                    if fg_mask.any():
                        pred_ltrb = bbox_pred[b].permute(1, 2, 0).reshape(-1, 4)[fg_mask]
                        tgt_ltrb = reg_target.reshape(-1, 4)[fg_mask]
                        batch_reg_loss += giou_loss(
                            pred_ltrb.float(), tgt_ltrb
                        ).item()

                        pred_cent = centerness[b, 0].reshape(-1)[fg_mask]
                        tgt_cent = cent_target.reshape(-1)[fg_mask]
                        batch_cent_loss += F.binary_cross_entropy_with_logits(
                            pred_cent.float(), tgt_cent, reduction="mean"
                        ).item()

            val_loss_sum += (batch_cls_loss + batch_reg_loss + batch_cent_loss) / B
            val_batches += 1

    val_loss = val_loss_sum / max(val_batches, 1)
    elapsed = time.time() - t0

    print(
        f"Epoch {epoch}/{EPOCHS} -- "
        f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
        f"time={elapsed:.0f}s  lr={optimizer.param_groups[0]['lr']:.2e}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint = {
            "epoch": epoch,
            "fpn_state_dict": detector.fpn.state_dict(),
            "head_state_dict": detector.head.state_dict(),
            "val_loss": val_loss,
        }
        torch.save(checkpoint, save_path)
        print(f"  Saved best checkpoint -> {save_path}")

print(f"\nTraining complete. Best val_loss={best_val_loss:.4f}")
print(f"Checkpoint saved to: {save_path}")

In [ ]:
# =========================================================================
# Download the trained checkpoint
# =========================================================================
# Option 1: Download directly from Colab
from google.colab import files
files.download(save_path)

# Option 2: The checkpoint is already on Google Drive at:
#   My Drive/VisionPDF_checkpoints/sam_doclaynet_head.pt
# You can download it from drive.google.com any time.